In [1]:
# Check GPU is working
!nvidia-smi

# Find the exact torch + CUDA version (needed for PyG)
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Fri Oct  9 19:12:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
# Modern PyG doesn't need torch-scatter/sparse to be compiled from source
# Just install torch-geometric directly — works on Colab with no build step

!pip install -q torch-geometric

!pip install -q transformers datasets rapidfuzz networkx rdflib \
    pandas numpy scikit-learn scipy rdkit pubchempy \
    pyyaml omegaconf matplotlib seaborn plotly \
    tqdm loguru rich python-dotenv pytest


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.8 MB/s eta 0:00:00


In [10]:
import torch_geometric
import torch
print("PyG:", torch_geometric.__version__)
print("CUDA:", torch.cuda.is_available())
print("✅ Ready!")


PyG: 2.8.0.post1
CUDA: True
✅ Ready!


In [11]:
import os

!git clone https://github.com/jiya2000/Reliability-Conditioned-Herb-Drug-Interaction-Prediction.git

# Move into the project folder
os.chdir("Reliability-Conditioned-Herb-Drug-Interaction-Prediction")

# Create all required folders
!mkdir -p logs results/figures checkpoints/ablation data/processed data/raw

# Confirm everything is there
print("✅ Working directory:", os.getcwd())
!ls


Cloning into 'Reliability-Conditioned-Herb-Drug-Interaction-Prediction'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 211 (delta 82), reused 181 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (211/211), 7.41 MiB | 5.86 MiB/s, done.
Resolving deltas: 100% (82/82), done.
✅ Working directory: /content/Reliability-Conditioned-Herb-Drug-Interaction-Prediction/Reliability-Conditioned-Herb-Drug-Interaction-Prediction
 app					  README.md
 checkpoints				  requirements_core.txt
 configs				  requirements.txt
 create_pdf.py				  results
 data					  Review_1_Report_Final.md
 docs					  Review_1_Report_Final.pdf
 generate_architecture_diagram.py	  Review_1_Report.md
'HDI_Final_Implementation_Plan (1).md'	  Review_1_Report.pdf
 implementation_plan.md			  run_experiments.py
 LICENSE				 'Screenshot 2026-07-23 230057.png'
 LITERATURE_REVIEW_AND_PROJECT_GUIDE.md   src
 logs					  

In [12]:
!python run_experiments.py \
    --mode synthetic \
    --epochs 30 \
    --seeds 42 123 456 \
    --hidden-dim 128 \
    --num-heads 4 \
    --batch-size 64


19:54:31 | INFO     | ======================================================================
19:54:31 | INFO     | HDI PREDICTION — FULL EXPERIMENT PIPELINE
19:54:31 | INFO     | ======================================================================
19:54:31 | INFO     | Mode: synthetic
19:54:31 | INFO     | Epochs: 30
19:54:31 | INFO     | Seeds: [42, 123, 456]
19:54:31 | INFO     | Gating: multiplicative
19:54:31 | INFO     | Device: auto
19:54:31 | INFO     | ======================================================================
19:54:31 | INFO     | Building data pipeline (mode=synthetic)...
19:54:31 | INFO     | Building data pipeline (mode=synthetic)...
19:54:31 | INFO     | Generating synthetic knowledge graph data...
19:54:31 | INFO     | Synthetic data generated: 229 nodes (110 drugs, 69 herbs, 50 targets), 2400 edges, 800 positive interaction edges (train=640, val=80, test=80)
19:54:31 | INFO     | HDIDataset: 640 positive edges, 3840 total samples (ratio 1:5)
19:54:31 | INFO

In [13]:
# Print results table
with open("results/paper_tables.md") as f:
    print(f.read())

# Also print raw JSON
import json
with open("results/ablation_results.json") as f:
    data = json.load(f)

print("\n📊 Key metrics:")
for variant, metrics in data.get("results", {}).items():
    if isinstance(metrics, dict):
        auc = metrics.get("auc_roc", metrics.get("auc", "N/A"))
        print(f"  {variant}: AUC = {auc}")


# Experimental Results — HDI Prediction

**Generated**: 2026-10-09 20:39:04
**Data mode**: synthetic
**Seeds**: [42, 123, 456]
**Epochs**: 30
**Gating mode**: multiplicative

## Table 1: Ablation Study Results

| Variant | AUC-ROC | AUC-PR | Precision | Recall | F1 | MRR | Hits@10 | Params |
|---------|---------|--------|-----------|--------|-----|-----|---------|--------|
| (a) GNN-only | 0.7568±0.0072 | 0.3163±0.0020 | 0.3645±0.1423 | 0.0917±0.0328 | 0.1388±0.0383 | 0.6043±0.0217 | 1.0000±0.0000 | 1,018,369 |
| (b) Unconditioned | 0.7554±0.0014 | 0.3137±0.0016 | 0.3379±0.0639 | 0.0583±0.0118 | 0.0974±0.0127 | 0.6152±0.0097 | 1.0000±0.0000 | 1,199,617 |
| (c) **Full R-conditioned (Ours)** | 0.7538±0.0025 | 0.3071±0.0114 | 0.3587±0.0906 | 0.0583±0.0156 | 0.1003±0.0265 | 0.5772±0.0102 | 1.0000±0.0000 | 1,208,359 |

## Table 2: Critical Comparison — (b) vs (c)

The (b)-vs-(c) gap is the evidentiary core of the paper and patent.

| Metric | Unconditioned (b) | Full Model (c) | Δ (c−b) | I

In [14]:
from google.colab import files
import shutil

# Zip the results folder
shutil.make_archive("/content/hdi_results", "zip",
                    "/content/Reliability-Conditioned-Herb-Drug-Interaction-Prediction/results")

# Download it
files.download("/content/hdi_results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Copy results to Drive
shutil.copytree(
    "results",
    "/content/drive/MyDrive/HDI_Patent_Results",
    dirs_exist_ok=True
)

print("✅ Saved to Google Drive → HDI_Patent_Results/")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Saved to Google Drive → HDI_Patent_Results/
